In [1]:
# This notebook assumes that in the same directory this notebook is being run, there is the following folder structure:
# cropped/
# cropped/no_sliding/
# cropped/sliding/
#
# In this cropped/ folder there must be cropped videos in its two sub-folders. Although this currently assumes the names of the
# videos are id.mp4, this doesn't neccessarily have to be true. This notebook also does NOT rely on any .csv files, although it
# does create results.csv which is an easy way to view the performance of the trained model. It will store the images it uses
# for training the model in var/, a newly created directory (it will create it if it doesn't exist already)
#
# To train a model, one can simply run the cells, ensuring the environment has the environment supplied by notebook_reqs.txt.
# Please consult https://janakiev.com/blog/jupyter-virtual-envs/ on how to add virtual environments to Jupyter.
#
# Note: Some variables are hardcoded, please read the comments throughout this notebook to see what these values do and how to
# modify them.

In [2]:
import numpy as np
import tensorflow as tf
import cv2
from utils import refresh_folder
import pandas as pd

In [ ]:
no_sliding_folder = 'cropped/no_sliding/'
sliding_folder = 'cropped/sliding/'

no_sliding_variance_folder = 'var/no_sliding/'
sliding_variance_folder = 'var/sliding/'
refresh_folder('var/')

refresh_folder(no_sliding_variance_folder)
refresh_folder(sliding_variance_folder)

sliding_images = []
no_sliding_images = []

var_sliding_info = []
var_no_sliding_info = []

import os

# Observe that resize below is (90, 90) - this can be modified to alter the transformation performed on
# each of the frames on every video.
def write_variance_images(var_folder_str, original_folder_str, empty_lst, info_lst):
    for filename in os.listdir(original_folder_str):
        cap = cv2.VideoCapture(original_folder_str + filename)
        
        if (cap.isOpened()== False):
            print("Error opening video stream or file")
        
        grey_clip = []
        while(cap.isOpened()):
            ret, frame = cap.read()
            
            if ret == True:
                grey_image = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                hw = grey_image.shape
                height = hw[0]
                width = hw[1]
                resize = (90, 90)
                new_width = resize[0]
                
                height_resize = int((height / width) * new_width)
                pad_top = int((tuple(resize)[1] - height_resize) / 2)
                pad_bottom = tuple(resize)[1] - height_resize - pad_top
                
                frame = cv2.resize(grey_image, (new_width, height_resize))
                grey_image = cv2.copyMakeBorder(frame, pad_top, pad_bottom, 0, 0, cv2.BORDER_CONSTANT, value=0)
                grey_clip.append(grey_image)
            else:
                break
            
        grey_clip = np.array(grey_clip)
        variance_image = np.var(grey_clip, axis=0)
        variance_image = (variance_image / np.max(variance_image.flatten())) * 255
        variance_image = variance_image.astype(np.int32)
        variance_image = variance_image.reshape((90, 90, 1))
        as_tensor = tf.convert_to_tensor(variance_image)
        converted = tf.image.grayscale_to_rgb(as_tensor)
        np.savez(var_folder_str + filename.split('.')[0] + '.npz', x=converted.numpy())
        path = var_folder_str + filename.split('.')[0] + '.npz'
        info_lst.append([path.split('/')[2].split('.')[0], path])
        cv2.imwrite(var_folder_str + filename.split('.')[0] + '.png', converted.numpy())
        empty_lst.append(variance_image)
            

write_variance_images(no_sliding_variance_folder, no_sliding_folder, no_sliding_images, var_no_sliding_info)
write_variance_images(sliding_variance_folder, sliding_folder, sliding_images, var_sliding_info)

sliding_images = np.array(sliding_images)
no_sliding_images = np.array(no_sliding_images)

In [ ]:
var_sliding_df = pd.DataFrame({'id':[x[0] for x in var_sliding_info], 'var_path':[x[1] for x in var_sliding_info]})
var_no_sliding_df = pd.DataFrame({'id':[x[0] for x in var_no_sliding_info], 'var_path':[x[1] for x in var_no_sliding_info]})
var_no_sliding_df

In [ ]:
def h(s):
    if s.split('/')[1][0] == 'n':
        return 0
    else:
        return 1

In [ ]:
var_no_sliding_df['label'] = var_no_sliding_df['var_path'].apply(h)
var_sliding_df['label'] = var_sliding_df['var_path'].apply(h)
var_sliding_df

In [ ]:
both_dfs = pd.concat([var_no_sliding_df, var_sliding_df])
both_dfs = both_dfs.sample(frac=1)
both_dfs

In [ ]:
# This 2200 is hardcoded to make a desired train-test split, this can easily be changed
train_df = both_dfs[:2200]
test_df = both_dfs[2200:]
train_df['label'].value_counts(), test_df['label'].value_counts()

In [ ]:
from tensorflow.python.ops import random_ops

# Note again here that (90, 90) is hardcoded
def parse_tf(filename, label):
    shape = (90, 90, 3)
    pic, label = tf.numpy_function(parse_fn, [filename, label], (tf.float32, tf.float32))
    pic.set_shape(shape)
    tf.ensure_shape(pic, shape)
    label.set_shape((1,))
    tf.ensure_shape(label, (1,))
    return pic, label

def parse_fn(filename, label):
    with np.load(filename, allow_pickle=True) as loaded_file:
        pic = loaded_file['x']
    pic = tf.cast(pic, tf.float32)
    return pic, (1 - tf.one_hot(label, 1))

def random_flip_left_right(x):

    r = random_ops.random_uniform([], 0, 1)
    if r < 0.5:
        x = tf.image.flip_left_right(x)
    return x

# Note again here that (90, 90) is hardcoded
def random_shift_clip(x):
    r = random_ops.random_uniform([], 0, 1)
    if r < cfg['TRAIN']['PARAMS']['AUGMENTATION_CHANCE']:
        h, w = 90, 90 
        dx = random_ops.random_uniform([], -.1, .1)
        dy = random_ops.random_uniform([], -.2, .2)
        translations = [dx, dy]
        x = tfa.image.translate(x, translations)
    return x

def fix(x):
    return tf.clip_by_value(x, 0, 1)

def augment_clip(x):
    x = x / 255
    x = tf.image.random_brightness(x, max_delta=0.3)
    x = tf.image.random_contrast(x, 0.7, 1)
    x = random_flip_left_right(x)
    x = fix(x)
    return x

def prepare(ds, augment=False):
    shuffle_val = len(ds)
    auto = tf.data.AUTOTUNE
    
    ds = ds.shuffle(shuffle_val)
    ds = ds.map(parse_tf, num_parallel_calls=auto)
    ds = ds.batch(2, num_parallel_calls=auto)
    
    if augment:
        ds = ds.map(lambda x, y: (augment_clip(x), y), num_parallel_calls=auto)
    
    ds = ds.prefetch(buffer_size=auto)
    return ds

In [ ]:
import tensorflow as tf

train_set = tf.data.Dataset.from_tensor_slices((train_df['var_path'].tolist(), train_df['label']))
test_set = tf.data.Dataset.from_tensor_slices((test_df['var_path'].tolist(), test_df['label']))
train_set, test_set

In [ ]:
train_set = prepare(train_set, augment=True)
test_set = prepare(test_set)
train_set, test_set

In [ ]:
from tensorflow.keras.metrics import Precision, Recall, AUC, TrueNegatives, TruePositives, FalseNegatives, FalsePositives, Accuracy

# Defining Binary Classification Metrics
metrics = ['accuracy', AUC(name='auc')]
metrics += [Precision(), Recall()]
metrics += [TrueNegatives(), TruePositives(), FalseNegatives(), FalsePositives()]

In [ ]:
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import L2

# Note again here that (90, 90) is hardcoded
model = Sequential([])
model.add(layers.Input((90, 90, 3)))
model.add(layers.Conv2D(32, (3,3), activation='relu', kernel_regularizer=L2(0.01), bias_regularizer=L2(0.01)))
model.add(layers.AveragePooling2D((2,2)))
model.add(layers.BatchNormalization())
model.add(layers.Conv2D(64, (3,3), activation='relu', kernel_regularizer=L2(0.01), bias_regularizer=L2(0.01)))
model.add(layers.AveragePooling2D((2,2)))
model.add(layers.BatchNormalization())
model.add(layers.Conv2D(128, (3,3), activation='relu', kernel_regularizer=L2(0.01), bias_regularizer=L2(0.01)))
model.add(layers.AveragePooling2D((2,2)))
model.add(layers.BatchNormalization())
model.add(layers.GlobalAveragePooling2D())
model.add(layers.Dense(32, activation='relu', kernel_regularizer=L2(0.01), bias_regularizer=L2(0.01)))
model.add(layers.Dropout(0.4))
model.add(layers.Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0001), metrics=metrics)

from tensorflow.keras.callbacks import ModelCheckpoint

cp = ModelCheckpoint('model/', save_best_only=True)

num_neg, num_pos = len(train_df[train_df['label']==0]), len(train_df[train_df['label']==1])

# These class weights are currently hardcoded to these values, but the above calculations can be used to perform an automated
# calculation (not neccessarily better)
class_weight = {0: 10,
                1: 1}
class_weight

model.summary()

In [ ]:
res = model.fit(train_set, validation_data=test_set, epochs=100, callbacks=[cp], class_weight=class_weight)

In [ ]:
pd.DataFrame(res.history).to_csv('results.csv')